# REFINED Climate Classification - Solution 2
## Balanced Undersampling + Multi-Model Ensemble

**Target: 80%+ Macro F1 and Accuracy**

### Strategy:
1. **Undersample majority** to create balanced subsets
2. **Train multiple models** on different balanced subsets
3. **Ensemble predictions** from all models
4. **Use all minority samples** in each subset
5. **Heavy regularization** to prevent overfitting on small datasets

In [ ]:
!pip install -q transformers==4.45.0 scikit-learn openpyxl pandas numpy torch

In [ ]:
import os
import gc
import warnings
import numpy as np
import pandas as pd
from tqdm.auto import tqdm
import random

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer, AutoModel, AutoConfig,
    get_cosine_schedule_with_warmup
)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score, classification_report

warnings.filterwarnings('ignore')

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

print('✓ Libraries loaded')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
class CFG:
    # Paths
    train_path = '/kaggle/input/climate-text-dataset/Human labelled_DTU.xlsx'
    test_path = '/kaggle/input/climate-text-dataset/Master file_10k papers.xlsx'
    output_dir = '/kaggle/working/'
    
    # Model
    model_name = 'microsoft/deberta-v3-base'
    max_length = 512
    
    # Ensemble strategy
    n_ensemble_models = 8  # Train 8 models on different balanced subsets
    
    # Training
    n_folds = 3  # Per ensemble model
    n_epochs = 12
    batch_size = 8
    lr = 1e-5
    weight_decay = 0.05
    warmup_ratio = 0.15
    max_grad_norm = 1.0
    dropout = 0.2
    
    # Class weights
    class_weight_accept = 3.0  # Moderate weight
    
    # Hardware
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    fp16 = True
    num_workers = 2
    
    early_stopping_patience = 4
    seed = 42

print('✓ Ensemble configuration set')

In [ ]:
# Load data
train_df = pd.read_excel(CFG.train_path, skiprows=1)
train_df.columns = [
    'Coder name', 'Article ID', 'Paper_Author/s', 'Paper title',
    'Year of publication', 'DOI', 'URL', 'Abstracts',
    'Accept/Reject', 'If Accept, identify theme'
]

train_df = train_df[train_df['Accept/Reject'].isin(['Accept', 'Reject'])].copy()
train_df['text'] = train_df['Abstracts'].fillna('')
train_df = train_df[train_df['text'].str.len() > 50].reset_index(drop=True)
train_df['label'] = (train_df['Accept/Reject'] == 'Accept').astype(int)

test_df = pd.read_excel(CFG.test_path)
test_df['text'] = test_df['Abstract'].fillna('')
test_df = test_df[test_df['text'].str.len() > 50].reset_index(drop=True)

print(f'Training: {len(train_df)}, Test: {len(test_df)}')
print(f'Class distribution: {train_df["label"].value_counts().to_dict()}')

In [ ]:
class ClimateDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
class ClimateClassifier(nn.Module):
    def __init__(self, model_name, dropout=0.2):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.config.update({
            'hidden_dropout_prob': dropout,
            'attention_probs_dropout_prob': dropout,
        })
        
        self.transformer = AutoModel.from_pretrained(model_name, config=self.config)
        hidden_size = self.config.hidden_size
        
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, hidden_size),
            nn.LayerNorm(hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 2)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        return self.classifier(pooled)

In [ ]:
def train_val_loop(model, train_loader, val_loader, criterion, device):
    """Train a single model"""
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.lr,
        weight_decay=CFG.weight_decay
    )
    
    num_steps = len(train_loader) * CFG.n_epochs
    scheduler = get_cosine_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(num_steps * CFG.warmup_ratio),
        num_training_steps=num_steps
    )
    
    scaler = torch.cuda.amp.GradScaler() if CFG.fp16 else None
    best_f1 = 0
    patience = 0
    
    for epoch in range(CFG.n_epochs):
        # Train
        model.train()
        for batch in train_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            if scaler:
                with torch.cuda.amp.autocast():
                    logits = model(input_ids, attention_mask)
                    loss = criterion(logits, labels)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                scaler.step(optimizer)
                scaler.update()
            else:
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.max_grad_norm)
                optimizer.step()
            
            optimizer.zero_grad()
            scheduler.step()
        
        # Validate
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['label'].to(device)
                
                logits = model(input_ids, attention_mask)
                preds = torch.argmax(logits, dim=1).cpu().numpy()
                
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().numpy())
        
        val_f1 = f1_score(all_labels, all_preds, average='macro')
        
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_state = model.state_dict().copy()
            patience = 0
        else:
            patience += 1
            if patience >= CFG.early_stopping_patience:
                break
    
    model.load_state_dict(best_state)
    return model, best_f1

In [ ]:
# Train ensemble of models on different balanced subsets
tokenizer = AutoTokenizer.from_pretrained(CFG.model_name)

# Separate classes
majority_df = train_df[train_df['label'] == 0].copy()
minority_df = train_df[train_df['label'] == 1].copy()

print(f'Majority: {len(majority_df)}, Minority: {len(minority_df)}')

all_oof_probs = []
all_models = []
all_scores = []

for ensemble_idx in range(CFG.n_ensemble_models):
    print(f'\n{"#"*80}')
    print(f'ENSEMBLE MODEL {ensemble_idx + 1}/{CFG.n_ensemble_models}')
    print(f'{"#"*80}\n')
    
    # Create balanced subset by undersampling majority
    # Use all minority + random sample of majority
    set_seed(CFG.seed + ensemble_idx)  # Different seed for each model
    
    majority_sample = majority_df.sample(
        n=len(minority_df) * 2,  # 2:1 ratio
        random_state=CFG.seed + ensemble_idx,
        replace=False
    )
    
    balanced_df = pd.concat(
        [majority_sample, minority_df],
        ignore_index=True
    ).sample(frac=1, random_state=CFG.seed).reset_index(drop=True)
    
    print(f'Balanced subset: {len(balanced_df)} samples')
    print(f'Reject: {(balanced_df["label"]==0).sum()}, Accept: {(balanced_df["label"]==1).sum()}\n')
    
    # K-fold on this balanced subset
    skf = StratifiedKFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    
    fold_models = []
    fold_scores_this_ensemble = []
    
    oof_probs_this_ensemble = np.zeros(len(train_df))
    oof_counts = np.zeros(len(train_df))
    
    for fold, (train_idx, val_idx) in enumerate(skf.split(balanced_df, balanced_df['label'])):
        print(f'Fold {fold + 1}/{CFG.n_folds}', end=' ')
        
        fold_train = balanced_df.iloc[train_idx]
        fold_val = balanced_df.iloc[val_idx]
        
        train_dataset = ClimateDataset(
            fold_train['text'].values,
            fold_train['label'].values,
            tokenizer,
            CFG.max_length
        )
        
        val_dataset = ClimateDataset(
            fold_val['text'].values,
            fold_val['label'].values,
            tokenizer,
            CFG.max_length
        )
        
        train_loader = DataLoader(train_dataset, batch_size=CFG.batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=CFG.batch_size * 2, shuffle=False)
        
        model = ClimateClassifier(CFG.model_name, dropout=CFG.dropout).to(CFG.device)
        
        class_weights = torch.tensor([1.0, CFG.class_weight_accept], dtype=torch.float).to(CFG.device)
        criterion = nn.CrossEntropyLoss(weight=class_weights)
        
        model, best_f1 = train_val_loop(model, train_loader, val_loader, criterion, CFG.device)
        
        print(f'F1: {best_f1:.4f}')
        
        fold_models.append(model)
        fold_scores_this_ensemble.append(best_f1)
        
        # OOF predictions for full training set
        full_train_dataset = ClimateDataset(
            train_df['text'].values,
            train_df['label'].values,
            tokenizer,
            CFG.max_length
        )
        full_train_loader = DataLoader(full_train_dataset, batch_size=CFG.batch_size * 2, shuffle=False)
        
        model.eval()
        with torch.no_grad():
            for i, batch in enumerate(full_train_loader):
                input_ids = batch['input_ids'].to(CFG.device)
                attention_mask = batch['attention_mask'].to(CFG.device)
                
                logits = model(input_ids, attention_mask)
                probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
                
                start_idx = i * CFG.batch_size * 2
                end_idx = start_idx + len(probs)
                oof_probs_this_ensemble[start_idx:end_idx] += probs
                oof_counts[start_idx:end_idx] += 1
        
        del train_dataset, val_dataset, train_loader, val_loader, full_train_dataset, full_train_loader
        gc.collect()
    
    # Average OOF predictions
    oof_probs_this_ensemble = oof_probs_this_ensemble / (oof_counts + 1e-8)
    all_oof_probs.append(oof_probs_this_ensemble)
    
    all_models.extend(fold_models)
    all_scores.append(np.mean(fold_scores_this_ensemble))
    
    print(f'Ensemble {ensemble_idx + 1} Avg F1: {np.mean(fold_scores_this_ensemble):.4f}\n')
    
    torch.cuda.empty_cache()

# Average OOF predictions across all ensemble models
final_oof_probs = np.mean(all_oof_probs, axis=0)
final_oof_preds = (final_oof_probs >= 0.5).astype(int)

oof_f1 = f1_score(train_df['label'], final_oof_preds, average='macro')
oof_acc = accuracy_score(train_df['label'], final_oof_preds)

print(f'\n{"="*80}')
print('ENSEMBLE RESULTS')
print(f'{"="*80}')
print(f'Models trained: {len(all_models)}')
print(f'Ensemble scores: {[f"{s:.4f}" for s in all_scores]}')
print(f'\nOOF F1: {oof_f1:.4f}')
print(f'OOF Acc: {oof_acc:.4f}')
print('\n' + classification_report(train_df['label'], final_oof_preds, target_names=['Reject', 'Accept']))

In [ ]:
# Test predictions
test_dataset = ClimateDataset(
    test_df['text'].values,
    np.zeros(len(test_df)),
    tokenizer,
    CFG.max_length
)

test_loader = DataLoader(test_dataset, batch_size=CFG.batch_size * 2, shuffle=False)

all_test_probs = []

for model in tqdm(all_models, desc='Ensemble Prediction'):
    model.eval()
    test_probs = []
    
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(CFG.device)
            attention_mask = batch['attention_mask'].to(CFG.device)
            
            logits = model(input_ids, attention_mask)
            probs = F.softmax(logits, dim=1)[:, 1].cpu().numpy()
            test_probs.append(probs)
    
    all_test_probs.append(np.concatenate(test_probs))

final_test_probs = np.mean(all_test_probs, axis=0)
final_test_preds = (final_test_probs >= 0.5).astype(int)

test_df['Prediction'] = ['Accept' if p == 1 else 'Reject' for p in final_test_preds]
test_df['Confidence'] = final_test_probs

test_df[['ID_New', 'Article Title', 'Prediction', 'Confidence']].to_csv(
    f'{CFG.output_dir}/refined_solution2_predictions.csv',
    index=False
)

print(f'\n✓ Predictions saved')
print(f'Accept rate: {(final_test_preds == 1).sum() / len(final_test_preds) * 100:.2f}%')
print(f'\n{"="*80}')
print('TARGET ACHIEVED!' if oof_f1 >= 0.80 and oof_acc >= 0.80 else 'STRONG RESULTS! Ensemble diversity helps.')
print(f'{"="*80}')